In [11]:
import pandas as pd

In [12]:
data = pd.read_csv('worlddata_top20_clusting.csv')
data

,Country,Agricultural Land( %),Land Area(Km2),Armed Forces size,Forested Area (%),GDP,Infant mortality,Life expectancy,Population,Population: Labor force participation (%),Unemployment rate,Urban_population,Cluster_n2
0,Afghanistan,58.1,652230,323000,2.1,19101353833,47.9,64.5,38041754,48.9,11.12,9797273,1
1,Albania,43.1,28748,9000,28.1,15278077447,7.8,78.5,2854191,55.7,12.33,1747593,0
2,Algeria,17.4,2381741,317000,0.8,169988236398,20.1,76.7,43053054,41.2,11.70,31510100,1
3,Angola,47.5,1246700,117000,46.3,94635415870,51.6,60.8,31825295,77.5,6.89,21061025,0
4,Argentina,54.3,2780400,105000,9.8,449663446954,8.8,76.5,44938712,61.3,9.79,41339571,1
5,Armenia,58.9,29743,49000,11.7,13672802158,11.0,74.9,2957731,55.6,16.99,1869848,0
6,Australia,48.2,7741220,58000,16.3,1392680589329,3.1,82.7,25766605,65.5,5.27,21844756,0
7,Austria,32.4,83871,21000,46.9,446314739528,2.9,81.6,8877067,60.7,4.67,5194416,0
8,Azerbaijan,57.7,86600,82000,14.1,39207000000,19.2,72.9,10023318,66.5,5.51,5616165,0
9,The Bahamas,1.4,13880,1000,51.4,12827000000,8.3,73.8,389482,74.6,10.36,323784,0


In [13]:
from sklearn.preprocessing import StandardScaler

In [15]:
numeric_features = data.select_dtypes(include=['float64', 'int64']).columns
numeric_features = numeric_features.drop('Cluster_n2')

scaler = StandardScaler()
data[numeric_features] = scaler.fit_transform(data[numeric_features])

print(data.head())

       Country  Agricultural Land( %)  Land Area(Km2)  Armed Forces size  \
0  Afghanistan               1.107945       -0.104929           2.472602   
1      Albania               0.315801       -0.457407          -0.730356   
2      Algeria              -1.041405        0.872829           2.411399   
3       Angola               0.548163        0.231148           0.371298   
4    Argentina               0.907268        1.098206           0.248892   

   Forested Area (%)       GDP  Infant mortality  Life expectancy  Population  \
0          -1.287994 -0.509255          1.812121        -1.620238    0.458384   
1          -0.048163 -0.521091         -0.586099         0.726718   -0.506512   
2          -1.349985 -0.042149          0.149515         0.424967    0.595802   
3           0.819719 -0.275422          2.033403        -2.240505    0.287920   
4          -0.920813  0.823651         -0.526293         0.391439    0.647509   

   Population: Labor force participation (%)  Unemployme

In [16]:
import statsmodels.api as sm
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [17]:
X = data.drop(columns=['Country', 'Cluster_n2'])
y = data['Cluster_n2']
X = sm.add_constant(X)

In [18]:
def forward_selection(X, y):
    initial_features = []
    remaining_features = list(X.columns)
    best_features = []
    while remaining_features:
        scores_with_candidates = []
        for candidate in remaining_features:
            features = initial_features + [candidate]
            X_train = X[features]
            model = sm.OLS(y, X_train).fit()
            scores_with_candidates.append((model.rsquared_adj, candidate))
        scores_with_candidates.sort()
        best_score, best_candidate = scores_with_candidates.pop()
        remaining_features.remove(best_candidate)
        initial_features.append(best_candidate)
        best_features.append(best_candidate)
        print(f'Added {best_candidate}, R-squared: {best_score}')
    return best_features

In [20]:
selected_features = forward_selection(X, y)
print("Selected features:", selected_features)

Added Armed Forces size, R-squared: 0.5144710364052147
Added const, R-squared: 0.6552918199384808
Added Urban_population, R-squared: 0.7640066890655485
Added Population: Labor force participation (%), R-squared: 0.784242865794348
Added GDP, R-squared: 0.7950493785038076
Added Agricultural Land( %), R-squared: 0.8007914747248286
Added Forested Area (%), R-squared: 0.8064885516071378
Added Life expectancy, R-squared: 0.8017036160549722
Added Infant mortality, R-squared: 0.7848011765371322
Added Population, R-squared: 0.76381500501735
Added Unemployment rate, R-squared: 0.7378923094957273
Added Land Area(Km2), R-squared: 0.7053251024716648
Selected features: ['Armed Forces size', 'const', 'Urban_population', 'Population: Labor force participation (%)', 'GDP', 'Agricultural Land( %)', 'Forested Area (%)', 'Life expectancy', 'Infant mortality', 'Population', 'Unemployment rate', 'Land Area(Km2)']


In [22]:
X_train, X_test, y_train, y_test = train_test_split(X[selected_features], y, test_size=0.3, random_state=42)

lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)

y_pred = lda.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8333333333333334
Classification Report:
               precision    recall  f1-score   support

           0       0.83      1.00      0.91         5
           1       0.00      0.00      0.00         1

    accuracy                           0.83         6
   macro avg       0.42      0.50      0.45         6
weighted avg       0.69      0.83      0.76         6



c:\Users\compro\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\compro\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\compro\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [25]:
numeric_features = usa.columns

scaler = StandardScaler()
usa[numeric_features] = scaler.fit_transform(usa[numeric_features])

new_data_selected = usa[selected_features]

lda = LinearDiscriminantAnalysis()
lda.fit(X[selected_features], y)  

new_pred = lda.predict(new_data_selected)
print("Prediction for the new data:", new_pred)

KeyError: "['const'] not in index"

In [29]:
data = pd.read_csv('worlddata_top20_clusting.csv')

numeric_features = data.select_dtypes(include=['float64', 'int64']).columns
numeric_features = numeric_features.drop('Cluster_n2')

# 정규화
scaler = StandardScaler()
data[numeric_features] = scaler.fit_transform(data[numeric_features])

# 설명변수(X)와 종속변수(y) 설정
X = data.drop(columns=['Country', 'Cluster_n2'])  # 'Country'와 'Cluster_n2'는 제외
y = data['Cluster_n2']

# 전진 선택법
import statsmodels.api as sm

def forward_selection(X, y):
    initial_features = []
    remaining_features = list(X.columns)
    best_features = []
    while remaining_features:
        scores_with_candidates = []
        for candidate in remaining_features:
            features = initial_features + [candidate]
            X_train = X[features]
            model = sm.OLS(y, X_train).fit()
            scores_with_candidates.append((model.rsquared_adj, candidate))
        scores_with_candidates.sort()
        best_score, best_candidate = scores_with_candidates.pop()
        remaining_features.remove(best_candidate)
        initial_features.append(best_candidate)
        best_features.append(best_candidate)
        print(f'Added {best_candidate}, R-squared: {best_score}')
    return best_features

selected_features = forward_selection(X, y)
print("Selected features:", selected_features)

# LDA 모델 학습
lda = LinearDiscriminantAnalysis()
lda.fit(X[selected_features], y)

Added Armed Forces size, R-squared: 0.5144710364052147
Added Urban_population, R-squared: 0.5900871913035939
Added Population: Labor force participation (%), R-squared: 0.5937033239732297
Added GDP, R-squared: 0.5881968777661641
Added Agricultural Land( %), R-squared: 0.5767624222398657
Added Forested Area (%), R-squared: 0.5629684914822739
Added Life expectancy, R-squared: 0.5381663736135824
Added Infant mortality, R-squared: 0.5005482766251557
Added Population, R-squared: 0.45555215695108264
Added Unemployment rate, R-squared: 0.4013499608809732
Added Land Area(Km2), R-squared: 0.3349801936630016
Selected features: ['Armed Forces size', 'Urban_population', 'Population: Labor force participation (%)', 'GDP', 'Agricultural Land( %)', 'Forested Area (%)', 'Life expectancy', 'Infant mortality', 'Population', 'Unemployment rate', 'Land Area(Km2)']


LinearDiscriminantAnalysis()

In [30]:
usa = pd.DataFrame({
    'Agricultural Land( %)': [44.4],
    'Land Area(Km2)': [9833517],
    'Armed Forces size': [1359000],
    'Forested Area (%)': [33.9],
    'GDP': [21427700000000],
    'Infant mortality': [5.6],
    'Life expectancy': [78.5],
    'Population': [328239523],
    'Population: Labor force participation (%)': [62.0],
    'Unemployment rate': [14.7],
    'Urban_population': [270663028]
})

In [31]:
usa[numeric_features] = scaler.transform(usa[numeric_features])

new_data_selected = usa[selected_features]

new_pred = lda.predict(new_data_selected)
print("Prediction for the new data:", new_pred)

Prediction for the new data: [1]
